| Student Name | Matric Number |
| :--- | :--- |
| [Name 1] | [Matric Number 1] |
| [Name 2] | [Matric Number 2] |
| [Name 3] | [Matric Number 3] |
| [Name 4] | [Matric Number 4] |


# Problem 1: Iterative PDDL Modelling with AI

A single lift is about to begin the evening service run, and you are responsible for specifying its behaviour. You will express the system in the Planning Domain Definition Language (PDDL), test the model, and extend it as new operating constraints are introduced. AI tools may be used to draft, explain, challenge, test, and refine the model.

We assess the resulting planning model rather than your prompts or conversation history. Regard every AI response as a hypothesis: expose its assumptions, trace it on specific states, and update it whenever a later requirement invalidates an earlier choice.

Begin with `0_env_walkthrough.ipynb`. The PDDL formulation is divided between two files:

1. `domain.pddl` contains the reusable vocabulary and transition rules.
2. `problem.pddl` instantiates that domain with objects, an initial state, and target goals for a given scenario.

## 1. Build the reusable domain

The `domain` file supplies the object categories (`types`), the facts that may hold (`predicates`), and the state transitions (`actions`). Because it is shared by every scenario, it must describe the lift system without assuming a particular set of passengers or floors.

### 1.1 Object types

The first model uses two object categories:

- `level`: a floor that the lift or a passenger may occupy.
- `person`: a passenger whose trip must be planned.

### 1.2 State predicates

Predicates encode the facts needed to distinguish one world state from another. Use the following fixed vocabulary for the initial lift model:

- `elevator_at(?l - level)`: locates the lift at level `?l`.
- `person_at(?p - person, ?l - level)`: locates passenger `?p` at level `?l`.
- `person_in_elevator(?p - person)`: states that passenger `?p` is aboard.
- `elevator_empty()`: states that nobody is currently aboard.
- `door_open(?l - level)`: records an open lift door at level `?l`.
- `adjacent_up(?from - level, ?to - level)`: links a floor to the one immediately above it.
- `adjacent_down(?from - level, ?to - level)`: links a floor to the one immediately below it.

### 1.3 Available actions

The walkthrough `0_env_walkthrough.ipynb` introduced six operations. Your PDDL actions must capture when each operation is legal and how it changes the state:

1. `open_door(?l - level)`: open the door at the lift's current floor.
2. `close_door(?l - level)`: close the door at that floor.
3. `load(?p - person, ?l - level)`: board a waiting passenger while the door is open and the lift is empty.
4. `unload(?p - person, ?l - level)`: let an onboard passenger leave through an open door.
5. `move_up(?from - level, ?to - level)`: travel to the adjacent floor above.
6. `move_down(?from - level, ?to - level)`: travel to the adjacent floor below.

### Task 1: Finish and inspect the domain model

Fill every `___` with valid PDDL. If AI supplies a first draft, require it to explain which invariant each precondition and effect preserves. The autograder examines variable identity and literal polarity, so a predicate name with the right arity is not enough.

In [ ]:
# COPY-FLAG-1-START

pddl_domain = """
(define (domain elevator)
  (:requirements :strips :typing :negative-preconditions)
  (:types level person)

  (:predicates
    (elevator_at ?l - level) ; The elevator is at a specific level
    (person_at ?p - person ?l - level) ; A person is at a specific level
    (person_in_elevator ?p - person) ; A person is in the elevator
    (elevator_empty) ; The elevator is empty
    (door_open ?l - level) ; The door is open at a specific level
    (adjacent_up ?from ?to - level) ; Defines that ?to is the level above ?from
    (adjacent_down ?from ?to - level) ; Defines that ?to is the level below ?from
  )

  ; move_up: The elevator can only move up one level
  (:action move_up
    :parameters (?from ?to - level)
    :precondition (and ___) ; FILL IN the precondition for move_up with one or more predicates
    :effect (and ___) ; FILL IN the effect for move_up with one or more predicates
  )

  ; move_down: The elevator can only move down one level
  (:action move_down
    :parameters (?from ?to - level)
    :precondition (and ___) ; FILL IN the precondition for move_down with one or more predicates
    :effect (and ___) ; FILL IN the effect for move_down with one or more predicates
  )

  ; open_door: Open the door without considering picking up people
  (:action open_door
    :parameters (?l - level)
    :precondition (and ___) ; FILL IN the precondition for open_door with one or more predicates
    :effect (and ___) ; FILL IN the effect for open_door with one or more predicates
  )

  ; close_door: Close the door
  (:action close_door
    :parameters (?l - level)
    :precondition (and ___) ; FILL IN the precondition for close_door with one or more predicates
    :effect (and ___) ; FILL IN the effect for close_door with one or more predicates
  )

  ; load: Pick up a person, requires the door to be open and the elevator to be empty
  (:action load
    :parameters (?p - person ?l - level)
    :precondition (and ___) ; FILL IN the precondition for load with one or more predicates
    :effect (and ___) ; FILL IN the effect for load with one or more predicates
  )

  ; unload: Drop off a person, requires the door to be open and the person to be in the elevator
  (:action unload
    :parameters (?p - person ?l - level)
    :precondition (and ___) ; FILL IN the precondition for unload with one or more predicates
    :effect (and ___) ; FILL IN the effect for unload with one or more predicates
  )
)
"""

# COPY-FLAG-1-END

with open("elevator_domain.pddl", "w", encoding="utf-8") as file:
    file.write(pddl_domain)

## 2. Diagnose plausible AI-generated fragments

Before adding more features, examine the three believable but faulty fragments below. For each one,
1. name the broken rule or invariant, 
2. construct a state or configuration that reveals the fault, and 
3. propose the smallest repair.

This diagnostic exercise has no separate marks.

### Fragment 1: loading while the door may be closed

```lisp
(:action load
  :parameters (?p - person ?l - level)
  :precondition (and (elevator_at ?l) (person_at ?p ?l) (elevator_empty))
  :effect (and (not (person_at ?p ?l)) (person_in_elevator ?p)
               (not (elevator_empty))))
```

### Fragment 2: deleting the destination during movement

```lisp
(:action move_up
  :parameters (?from ?to - level)
  :precondition (and (elevator_at ?from) (adjacent_up ?from ?to)
                     (not (door_open ?from)))
  :effect (and (not (elevator_at ?to)) (elevator_at ?from)))
```

### Fragment 3: using every passenger's start as their goal

```python
for person, (start, goal) in zip(persons, requests):
    pddl += f"(person_at {person} level{start})\n"
```

One productive follow-up is: *Construct the smallest counterexample to your proposal, then trace the state immediately before and after the failing action.*

### Record your diagnosis

For every fragment, note the violated invariant, a concrete counterexample, and a minimal correction. Summarise the technical finding yourself instead of pasting a model transcript. If you want to retain these notes in your submission, use `optional_ai_audit_notes` near the top of `1_PDDL_solution.py`. That field is optional, is not read by the autograder, and carries no marks.

- **Fragment 1:**
- **Fragment 2:**
- **Fragment 3:**

## 3. Generalise the problem generator

The starter scenario `0_env_walkthrough.ipynb` placed the lift at `level0` and gave every passenger that same destination `level0`. You will retain the fixed PDDL vocabulary while also supporting any valid initial elevator floor, multiple passengers starting from or going to the same floor, passenger-specific destinations, and passenger requests that are already satisfied.

### Task 2: Instantiate the fixed configuration schema

Accept configurations in the form below. Entry `requests[i]` describes `person{i + 1}` as `(start_floor, goal_floor)`.

```python
config = {
    "num_levels": 5,
    "elevator_start": 2,
    "requests": [(0, 4), (4, 1), (2, 2), (4, 0)],
}
```

Treat `validate_config` as the complete input contract. Hidden cases vary the documented values and combinations; they do not add fields or invent new rules.


In [ ]:
def validate_config(config):
    required = {"num_levels", "elevator_start", "requests"}
    if not isinstance(config, dict) or set(config) != required:
        raise ValueError(f"config must contain exactly these keys: {sorted(required)}")

    num_levels = config["num_levels"]
    elevator_start = config["elevator_start"]
    requests = config["requests"]
    if not isinstance(num_levels, int) or isinstance(num_levels, bool) or num_levels < 1:
        raise ValueError("num_levels must be a positive integer")
    if (not isinstance(elevator_start, int) or isinstance(elevator_start, bool)
            or not 0 <= elevator_start < num_levels):
        raise ValueError("elevator_start must name an existing floor")
    if not isinstance(requests, (list, tuple)):
        raise ValueError("requests must be a list or tuple of (start, goal) pairs")
    for request in requests:
        if not isinstance(request, (list, tuple)) or len(request) != 2:
            raise ValueError("each request must be a (start, goal) pair")
        start, goal = request
        if (not isinstance(start, int) or isinstance(start, bool)
                or not isinstance(goal, int) or isinstance(goal, bool)):
            raise ValueError("request floors must be integers")
        if not 0 <= start < num_levels or not 0 <= goal < num_levels:
            raise ValueError("request floors must name existing floors")


# COPY-FLAG-2-START

def generate_pddl_from_config(config, output_file):
    validate_config(config)
    num_levels = config["num_levels"]
    elevator_start = config["elevator_start"]
    requests = config["requests"]
    persons = [f"person{i + 1}" for i in range(len(requests))]

    pddl = "(define (problem elevator_problem)\n"
    pddl += "  (:domain elevator)\n"
    levels = " ".join(f"level{i}" for i in range(num_levels))
    pddl += "  (:objects\n"
    pddl += f"    {levels} - level\n"
    if persons:
        pddl += f"    {' '.join(persons)} - person\n"
    pddl += "  )\n\n"

    pddl += "  (:init\n"
    pddl += f"    (___________)\n"  # Use elevator_start
    pddl += "    (elevator_empty)\n"
    for person, (start, _goal) in zip(persons, requests):
        pddl += f"    (___________)\n"  # Place person at start
    for floor in range(num_levels - 1):
        pddl += f"    (___________)\n"  # Upward adjacency
        pddl += f"    (___________)\n"  # Downward adjacency
    pddl += "  )\n\n"

    pddl += "  (:goal\n    (and\n"
    for person, (_start, goal) in zip(persons, requests):
        pddl += f"      (___________)\n"  # Place person at their goal
    pddl += "    )\n  )\n)"

    with open(output_file, "w", encoding="utf-8") as file:
        file.write(pddl)
    return pddl

# COPY-FLAG-2-END

Run the test scenarios below in order. As you refine your generator to handle new scenarios and edge cases, ensure that previously passing configurations continue to work. `config_1` is the basic case, `config_2` places several passengers on one floor, `config_3` introduces distinct goals and a non-ground lift start, and `config_4`--`config_5` cover degenerate inputs. When requesting an AI revision, explicitly include the earlier requirements and regression cases.

In [ ]:
config_1 = {
    "num_levels": 3,
    "elevator_start": 0,
    "requests": [(2, 0)],
}
pddl_string_1 = generate_pddl_from_config(config_1, "elevator_problem1.pddl")

config_2 = {"num_levels": 4, "elevator_start": 0,
            "requests": [(3, 0), (3, 0), (1, 0)]}
pddl_string_2 = generate_pddl_from_config(config_2, "elevator_problem2.pddl")

config_3 = {"num_levels": 5, "elevator_start": 2,
            "requests": [(0, 4), (4, 1), (2, 2), (4, 0)]}
pddl_string_3 = generate_pddl_from_config(config_3, "elevator_problem3.pddl")

config_4 = {"num_levels": 1, "elevator_start": 0, "requests": [(0, 0), (0, 0)]}
pddl_string_4 = generate_pddl_from_config(config_4, "elevator_problem4.pddl")

config_5 = {"num_levels": 4, "elevator_start": 3, "requests": []}
pddl_string_5 = generate_pddl_from_config(config_5, "elevator_problem5.pddl")

## 4. Verify the generated models

Run the given scenarios with the supplied planner, but do not treat planner success as proof of correctness: an underspecified goal can also be solved easily. Inspect the emitted `:objects`, `:init`, and `:goal` sections directly.

For this model, the classical planner minimises the number of primitive actions. That objective does not capture passenger waiting or satisfaction; Problem 2 will introduce those concerns explicitly.

In [ ]:
from EleEnv.PDDL import PDDL_Parser

def check_generated_problem(config, problem_file):
    parser = PDDL_Parser()
    parser.parse_domain("elevator_domain.pddl")
    parser.parse_problem(problem_file)

    persons = [f"person{i + 1}" for i in range(len(config["requests"]))]
    expected_levels = {f"level{i}" for i in range(config["num_levels"])}
    assert set(parser.objects) <= {"level", "person"}
    actual_levels = parser.objects.get("level", [])
    actual_people = parser.objects.get("person", [])
    assert len(actual_levels) == len(expected_levels)
    assert len(actual_people) == len(persons)
    assert set(actual_levels) == expected_levels
    assert set(actual_people) == set(persons)

    expected_init = {
        ("elevator_at", f"level{config['elevator_start']}"),
        ("elevator_empty",),
    }
    expected_init.update(
        ("person_at", person, f"level{start}")
        for person, (start, _goal) in zip(persons, config["requests"])
    )
    for floor in range(config["num_levels"] - 1):
        expected_init.add(("adjacent_up", f"level{floor}", f"level{floor + 1}"))
        expected_init.add(("adjacent_down", f"level{floor + 1}", f"level{floor}"))

    expected_goal = {
        ("person_at", person, f"level{goal}")
        for person, (_start, goal) in zip(persons, config["requests"])
    }
    assert parser.state == frozenset(expected_init)
    assert parser.positive_goals == frozenset(expected_goal)

for config, path in [(config_1, "elevator_problem1.pddl"),
                     (config_2, "elevator_problem2.pddl"),
                     (config_3, "elevator_problem3.pddl"),
                     (config_4, "elevator_problem4.pddl"),
                     (config_5, "elevator_problem5.pddl")]:
    check_generated_problem(config, path)
print("All visible structure checks passed.")

In [ ]:
from EleEnv.PDDL import Planner

domain = "elevator_domain.pddl"
problems = ["elevator_problem1.pddl", "elevator_problem2.pddl",
            "elevator_problem3.pddl", "elevator_problem4.pddl",
            "elevator_problem5.pddl"]

for problem in problems:
    planner = Planner()
    plan = planner.solve(domain, problem)
    if plan is not None:
        print(f"{problem}: {len(plan)} actions")
        for act in plan:
            print(act.name, *act.parameters)
    else:
        raise AssertionError(f"No plan found for {problem}")
    print("-------------")

## 5. Add destinations and a configurable capacity

The initial model uses `elevator_empty`, so it cannot represent more than one onboard passenger. Replace that Boolean abstraction with an occupancy counter, and store each passenger's destination as an unchanging fact.

The extended configuration therefore introduces a `capacity` field:

```python
capacity_config = {
    "num_levels": 5,
    "elevator_start": 2,
    "capacity": 2,
    "requests": [(0, 4), (0, 1), (2, 2)],
}
```

### 5.1 Revised state vocabulary
Remove `elevator_empty`, then introduce an additional type `count` whose objects `c0, c1, ..., c{capacity}` stand for the possible
  numbers of passengers in the elevator.

We will also introduce the following predicates:
- `lift_count(?c - count)`: The elevator currently holds `?c` passengers. Exactly
  one `lift_count` fact holds at a time, starting from `(lift_count c0)`.
- `next_count(?current - count, ?next - count)`: A static fact linking each
  occupancy to the next one up (`c0` to `c1`, `c1` to `c2`, and so on). `load`
  moves one step along this chain and `unload` moves one step back. The problem
  file lists only `capacity` links, so when the elevator is full there is no link
  for `load` to use.
- `destination(?p - person, ?l - level)`: The level that passenger `?p` must be
  delivered to. It is set in the initial state and never changes.
- `reached(?p - person)`: Passenger `?p` has been delivered to their destination.

A passenger starting at the destination is initially marked `reached` and must never board.

Before filling the template, ask an AI tool for candidate `load` and `unload` actions and investigate these two recurring errors:

- If `unload` uses `(next_count ?current ?previous)`, does it decrement or increment the occupancy?
- If `load` omits `(not (reached ?p))`, can a delivered passenger board again while `reached` remains true? What could the final goal state then mean?

The investigation is formative and carries no separate score. You may record a concise conclusion in the optional, ungraded `optional_ai_audit_notes` field of `1_PDDL_solution.py`.

### Task 3: Finish the capacity-aware domain

Replace the `___` entries in `load` and `unload`, leaving every supplied type, predicate, action, and parameter list unchanged.


In [ ]:
# COPY-FLAG-3-START

pddl_domain_capacity = """
(define (domain elevator)
  (:requirements :strips :typing :negative-preconditions)
  (:types level person count)

  (:predicates
    (elevator_at ?l - level)
    (person_at ?p - person ?l - level)
    (person_in_elevator ?p - person)
    (destination ?p - person ?l - level)
    (reached ?p - person)
    (door_open ?l - level)
    (adjacent_up ?from ?to - level)
    (adjacent_down ?from ?to - level)
    (lift_count ?c - count)
    (next_count ?current ?next - count)
  )

  (:action move_up
    :parameters (?from ?to - level)
    :precondition (and (elevator_at ?from) (adjacent_up ?from ?to)
                       (not (door_open ?from)))
    :effect (and (not (elevator_at ?from)) (elevator_at ?to))
  )

  (:action move_down
    :parameters (?from ?to - level)
    :precondition (and (elevator_at ?from) (adjacent_down ?from ?to)
                       (not (door_open ?from)))
    :effect (and (not (elevator_at ?from)) (elevator_at ?to))
  )

  (:action open_door
    :parameters (?l - level)
    :precondition (and (elevator_at ?l) (not (door_open ?l)))
    :effect (and (door_open ?l))
  )

  (:action close_door
    :parameters (?l - level)
    :precondition (and (elevator_at ?l) (door_open ?l))
    :effect (and (not (door_open ?l)))
  )

  (:action load
    :parameters (?p - person ?l - level ?current ?next - count)
    :precondition (and ___)
    :effect (and ___)
  )

  (:action unload
    :parameters (?p - person ?l - level ?previous ?current - count)
    :precondition (and ___)
    :effect (and ___)
  )
)
"""

# COPY-FLAG-3-END

with open("elevator_domain_capacity.pddl", "w", encoding="utf-8") as file:
    file.write(pddl_domain_capacity)

## 6. Instantiate the capacity-aware model

### Task 4: Finish the extended generator

Generate the `count` chain together with passenger locations, destinations, initially completed requests, and goal facts. There must be exactly `capacity` links between `c0` and `c{capacity}`. Express completion with `reached` rather than only `person_at`, ensuring that delivery is credited solely at the declared destination.


In [ ]:
def validate_capacity_config(config):
    required = {"num_levels", "elevator_start", "capacity", "requests"}
    if not isinstance(config, dict) or set(config) != required:
        raise ValueError(f"config must contain exactly these keys: {sorted(required)}")
    validate_config({key: config[key] for key in ("num_levels", "elevator_start", "requests")})
    capacity = config["capacity"]
    if not isinstance(capacity, int) or isinstance(capacity, bool) or capacity < 1:
        raise ValueError("capacity must be a positive integer")

# COPY-FLAG-4-START

def generate_capacity_pddl_from_config(config, output_file):
    validate_capacity_config(config)
    num_levels = config["num_levels"]
    elevator_start = config["elevator_start"]
    capacity = config["capacity"]
    requests = config["requests"]
    persons = [f"person{i + 1}" for i in range(len(requests))]
    counts = [f"c{i}" for i in range(capacity + 1)]

    pddl = "(define (problem elevator_capacity_problem)\n"
    pddl += "  (:domain elevator)\n"
    pddl += "  (:objects\n"
    pddl += f"    {' '.join(f'level{i}' for i in range(num_levels))} - level\n"
    if persons:
        pddl += f"    {' '.join(persons)} - person\n"
    pddl += f"    {' '.join(counts)} - count\n  )\n\n"

    pddl += f"  (:init\n    (elevator_at level{elevator_start})\n    (lift_count c0)\n"
    for floor in range(num_levels - 1):
        pddl += f"    (adjacent_up level{floor} level{floor + 1})\n"
        pddl += f"    (adjacent_down level{floor + 1} level{floor})\n"
    for count in range(capacity):
        pddl += f"    (___________)\n"
    for person, (start, goal) in zip(persons, requests):
        pddl += f"    (___________)\n"
        pddl += f"    (___________)\n"
        if start == goal:
            pddl += f"    (___________)\n"
    pddl += "  )\n\n  (:goal\n    (and\n"
    for person in persons:
        pddl += f"      (___________)\n"
    pddl += "    )\n  )\n)"

    with open(output_file, "w", encoding="utf-8") as file:
        file.write(pddl)
    return pddl

# COPY-FLAG-4-END

In [ ]:
capacity_configs = [
    {"num_levels": 4, "elevator_start": 0, "capacity": 1,
     "requests": [(3, 0), (1, 2)]},
    {"num_levels": 5, "elevator_start": 3, "capacity": 2,
     "requests": [(1, 4), (1, 0), (3, 3)]},
    {"num_levels": 1, "elevator_start": 0, "capacity": 3,
     "requests": [(0, 0), (0, 0)]},
    {"num_levels": 6, "elevator_start": 5, "capacity": 4,
     "requests": []},
]

for index, config in enumerate(capacity_configs, start=1):
    path = f"elevator_capacity_problem{index}.pddl"
    generate_capacity_pddl_from_config(config, path)
    parser = PDDL_Parser()
    parser.parse_domain("elevator_domain_capacity.pddl")
    parser.parse_problem(path)
    persons = [f"person{i + 1}" for i in range(len(config["requests"]))]
    expected_levels = {f"level{i}" for i in range(config["num_levels"])}
    expected_counts = {f"c{i}" for i in range(config["capacity"] + 1)}
    assert set(parser.objects) <= {"level", "person", "count"}
    assert set(parser.objects.get("level", [])) == expected_levels
    assert set(parser.objects.get("person", [])) == set(persons)
    assert set(parser.objects.get("count", [])) == expected_counts

    expected_init = {
        ("elevator_at", f"level{config['elevator_start']}"),
        ("lift_count", "c0"),
    }
    for floor in range(config["num_levels"] - 1):
        expected_init.add(("adjacent_up", f"level{floor}", f"level{floor + 1}"))
        expected_init.add(("adjacent_down", f"level{floor + 1}", f"level{floor}"))
    for count in range(config["capacity"]):
        expected_init.add(("next_count", f"c{count}", f"c{count + 1}"))
    for person, (start, goal) in zip(persons, config["requests"]):
        expected_init.add(("person_at", person, f"level{start}"))
        expected_init.add(("destination", person, f"level{goal}"))
        if start == goal:
            expected_init.add(("reached", person))
    assert parser.state == frozenset(expected_init)
    assert parser.positive_goals == frozenset(
        ("reached", f"person{i + 1}") for i in range(len(config["requests"]))
    )
print("All visible capacity structure checks passed.")

## Submission and grading (2 points)

Submit `1_PDDL_solution.py`.

Only `COPY-FLAG-3` and `COPY-FLAG-4` are graded: the capacity-aware domain and capacity-aware problem generator are worth 1 point each, for 2 points in total. 

`COPY-FLAG-1` and `COPY-FLAG-2` must still be completed so that the foundational notebook examples, parser checks, and planner workflow run successfully, but they carry no marks. Notebook text and the `optional_ai_audit_notes` field are not read by the autograder; the optional field may be left blank without affecting your score.